In [0]:
%run ../Streaming_source/NB_generate_streaming_data

In [0]:
def load_to_gold_without_joins(columns:list, target_table_name:str, source_name:str=dbutils.widgets.get("source_name")):
    """
    Loads the silver table to the gold table

    Args:
        source_name (str, optional): name of the silver table
        columns (list): list of columns to be used in the gold table
        target_table_name (str): name of the gold table
    """

    main_df = spark.table(f"workspace.silver.{source_name}_main")

    dim_hotels_df = main_df.select(*columns)

    dim_hotels_df.write.mode("overwrite").saveAsTable(f"workspace.gold.{source_name}_{target_table_name}")

In [0]:
def generate_bronze_streaming_data(table_name:str, num_files:int=10, num_rows_per_file:int=1000,source_name:str=dbutils.widgets.get("source_name")):
    """
    Generates streaming data for the given table and source name

    Args:
        source_name (str, optional): The name of the source for the data
        table_name (str): The name of the table for the data
        num_files (int): The number of files to generate
        num_rows_per_file (int): The number of rows per file
    Returns:
        landing_path (str): The path to the landing directory
        checkpoint_path (str): The path to the checkpoint directory
        schema_path (str): The path to the schema directory
    """

    landing_path, checkpoint_path, schema_path = generate_streaming_data(source_name=source_name, table_name=table_name, num_files=num_files, num_rows_per_file=1000)

    bronze_df = (
        spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .option("cloudFiles.schemaLocation", schema_path) \
        .option("cloudFiles.inferColumnTypes", "true") \
        .load(landing_path) \
        .writeStream \
        .format("delta") \
        .option("checkpointLocation", checkpoint_path) \
        .trigger(once=True) \
        .table(f"workspace.bronze.{source_name}_{table_name}")
    )